In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import jax
import jax.numpy as np
import jax.random as jr
from jax import lax

import numpy as onp
from functools import partial

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from typing import Callable, Dict, Optional, Tuple

In [ ]:
SEED = 0

# Reservoir parameters
eps_num = 0.01
eps_den = eps_num
gamma = 0.96
alpha = 0.5


# Parameters for sequence generation
n_seq = 50 # only when doing batches, useful to compute statistics
n_steps = 1000
n_features = 128
p_input = 0.15


# print options
np.set_printoptions(precision=4, suppress=True)

# Plot stuff
# Consistent binary colour map for raster plots.
BINARY_CMAP = ListedColormap(["#f0f0f0", "#2c7bb6"])   # white / blue

plt.rcParams.update({
    "figure.dpi":      120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size":       10,
})

In [ ]:
# ── IID Bernoulli ────────────────────────────────────────────────────────────

def gen_iid_bernoulli(
    T: int, C_in: int, p: float,
    key: jr.PRNGKey, **_
):
    key, subkey = jr.split(key)
    return (jr.uniform(subkey, (T, C_in)) < p).astype(np.float32)


# ── Markov chain ─────────────────────────────────────────────────────────────

def gen_markov(
    T: int, C_in: int, p: float,
    key: jr.PRNGKey,
    p_on:  float = 0.15,
    p_off: float = 0.40,
    **_
):
    stat = p_on / (p_on + p_off)

    key, k_init, k_loop = jr.split(key, 3)
    z0 = jr.uniform(k_init, (C_in,)) < stat

    def step(z_prev, key):
        k1, k2 = jr.split(key)
        stay_on = z_prev & (jr.uniform(k1, (C_in,)) >= p_off)
        turn_on = (~z_prev) & (jr.uniform(k2, (C_in,)) < p_on)
        z = stay_on | turn_on
        return z, z

    keys = jr.split(k_loop, T - 1)
    _, Z_tail = lax.scan(step, z0, keys)

    Z = np.concatenate([z0[None, :], Z_tail], axis=0)
    return Z.astype(np.float32)


# ── Bursty generator ─────────────────────────────────────────────────────────

def gen_bursty(
    T: int, C_in: int, p: float,
    key: jr.PRNGKey,
    burst_len: int   = 10,
    burst_p:   float = 0.40,
    quiet_p:   float = 0.05,
    **_
):
    key, k_phase, k_ttl, k_loop = jr.split(key, 4)

    phase0 = jr.uniform(k_phase, (C_in,)) < 0.5
    ttl0   = jr.randint(k_ttl, (C_in,), 1, burst_len * 2)

    def step(carry, key):
        phase, ttl = carry
        k1, k2 = jr.split(key)

        p_t = np.where(phase, burst_p, quiet_p)
        z   = jr.uniform(k1, (C_in,)) < p_t

        ttl = ttl - 1
        flip = ttl <= 0

        phase = np.where(flip, ~phase, phase)
        new_ttl = jr.randint(k2, (C_in,), 1, burst_len * 2)
        ttl = np.where(flip, new_ttl, ttl)

        return (phase, ttl), z

    keys = jr.split(k_loop, T)
    (_, _), Z = lax.scan(step, (phase0, ttl0), keys)

    return Z.astype(np.float32)


# ── Spatial wave ─────────────────────────────────────────────────────────────

def gen_spatial_wave(
    T: int, C_in: int, p: float,
    key: jr.PRNGKey,
    speed: float = 2.0,
    width: float = 0.15,
    **_
):
    units = np.arange(C_in)

    def step(carry, t):
        key = carry
        key, subkey = jr.split(key)

        centre = (speed * t) % C_in
        dist = np.minimum(
            np.abs(units - centre),
            C_in - np.abs(units - centre)
        )

        prob = np.exp(-0.5 * (dist / (width * C_in)) ** 2)
        prob = prob / prob.sum() * (p * C_in)
        prob = np.clip(prob, 0.0, 1.0)

        z = jr.uniform(subkey, (C_in,)) < prob
        return key, z

    ts = np.arange(T)
    _, Z = lax.scan(step, key, ts)

    return Z.astype(np.float32)


# ── Registry ─────────────────────────────────────────────────────────────────

GENERATORS = {
    "iid":          gen_iid_bernoulli,
    "markov":       gen_markov,
    "bursty":       gen_bursty,
    "spatial_wave": gen_spatial_wave,
}

In [ ]:
def make_batched_generator(gen_fn, B, T, C_in):

    @jax.jit
    def batched(p, key, **kwargs):
        keys = jr.split(key, B)
        vmapped = jax.vmap(
            lambda k: gen_fn(T, C_in, p, k, **kwargs)
        )
        return vmapped(keys)

    return batched


# ── Create batched + JIT versions ────────────────────────────────────────────

BATCHED_GENERATORS = {
    name: make_batched_generator(fn, B=n_seq, T=n_steps, C_in=n_features)
    for name, fn in GENERATORS.items()
}

In [ ]:
# State initialization
def init_state(key, z):
    # z asssumed of shape (batch, time, features)
    h = np.zeros((*z.shape[:-2], z.shape[-1]), dtype=z.dtype)
    z_trace = np.zeros((*z.shape[:-2], z.shape[-1]), dtype=z.dtype)
    h_trace = np.zeros((*z.shape[:-2], z.shape[-1]), dtype=z.dtype)
    state = {
        "h": h,
        "z_trace": z_trace,
        "h_trace": h_trace,
    }
    return state

# Single step Mi update rule
def mi_update_rule(carry, input):
    
    state = carry
    z = input
    h_prev = state["h"]
    z_avg = state["z_trace"]
    h_avg = state["h_trace"]

    # # # Selection
    # z_avg = jnp.clip(params["z_pos_avg"], a_min=eps)
    # s_avg = jnp.clip(params["s_avg"], a_min=eps)
    # z_mem = z.copy()
    # s_mem = s_prev.copy()

    def update_memory(crr, inp):
        h = crr
        # Update memory with InfoNCE-based selection mechanism
        pos_sz = z / ((h * z).sum(-1, keepdims=True) + eps_den)
        pos_ss = h_prev / ((h * h_prev).sum(-1, keepdims=True) + eps_den)
        neg_sz = z_avg / ((h * z_avg).sum(-1, keepdims=True) + eps_den)
        neg_ss = h_avg / ((h * h_avg).sum(-1, keepdims=True) + eps_den)
        score = (pos_sz - neg_sz) + (pos_ss  - neg_ss)
        # s = s + score
        # s = jnp.clip(s, a_min=0.0, a_max=1.0)
        h = (score>0).astype(np.float32)
        return h, h

    # initialize h0 from the XOR between the past state and the input
    h0 = h_prev # np.zeros_like(h_prev)
    # h0 = z * (1.0 - h_prev) + (1.0 - z) * h_prev
    h, _ = jax.lax.scan(update_memory, h0, None, length=2)

    # # compute mi terms
    # score_z = np.log((z + eps_num)/(z_trace_prev + eps_den))
    # score_h = np.log((h_prev + eps_num)/(h_trace_prev + eps_den))
    # # new h
    # h = (alpha * score_z + (1 - alpha) * score_h) > 0
    # h = h.astype(z.dtype)

    # update discounted traces
    h_trace = gamma * state["h_trace"] + (1 - gamma) * h
    z_trace = gamma * state["z_trace"] + (1 - gamma) * z

    state = {
        "h": h,
        "z_trace": z_trace,
        "h_trace": h_trace,
    }
    return state, state


# function to run over a (batched) sequence of inputs z
def forward_scan(key, z):

    # init state
    state0 = init_state(key, z)

    # move the time axis as first, needed by jax.lax.scan
    z = np.moveaxis(z, -2, 0)

    # scan over the sequences
    _, state = jax.lax.scan(mi_update_rule, state0, z,)

    # put time axis back
    state = jax.tree.map(lambda x : np.moveaxis(x, 0, -2), state)

    return state


In [ ]:
# ── Compute Metrics ─────────────────────────────────────────────────────────

def compute_metrics(
    z: np.ndarray,
    state,
    burn_in: int = 20,
) -> Dict[str, float]:
    """
    Compute scalar diagnostics for a CA trajectory.

    Parameters
    ----------
    z       : bool ndarray, shape (B, T, C_in)
    state       : dict with the state at every time step
    burn_in : int  Steps to discard before computing statistics.

    Returns
    -------
    metrics : dict with keys:
        mean_sparsity     — time-averaged fraction of active hidden units
        std_sparsity      — standard deviation of per-step sparsity
        mean_hamming_dist — mean Hamming distance between consecutive h(t), h(t+1)
        unit_activity_std — std of per-unit time-averaged activity (diversity)
        input_sparsity    — actual input sparsity (sanity check)
    """
    Z_ = z[..., burn_in:, :]
    H_ = state["h"][..., burn_in:, :]

    per_step_sparsity  = H_.mean(axis=-1)              # (T-burn_in,)
    consecutive_hamming = np.abs(
        H_[..., 1:, :] - H_[..., :-1, :]
    ).sum(axis=-1)                                     # (T-burn_in-1,)
    per_unit_activity  = H_.mean(axis=(0,1))              # (C_h,)

    return {
        "mean_sparsity":     float(per_step_sparsity.mean()),
        "std_sparsity":      float(per_step_sparsity.std()),
        "mean_hamming_dist": float(consecutive_hamming.mean()),
        "unit_activity_std": float(per_unit_activity.std()),
        "input_sparsity":    float(Z_.mean()),
    }


print("run_ca and compute_metrics defined.")

In [ ]:
# run a batch of sequence for each generator and print the statistics
for name, gen in BATCHED_GENERATORS.items():
    print(f"Running {name}")
    key = jax.random.PRNGKey(SEED)
    z = gen(
        p=p_input,
        key=key,
    )
    state = forward_scan(key, z)
    metrics = compute_metrics(z, state)
    print(f"  {name} metrics: {metrics}")

In [ ]:
# Create a plotly plot to visualize one trajectory per generator (raster plot/heatmap of input and hidden states)
k_gen_list = list(GENERATORS.keys())

fig = make_subplots(
    rows = 2, cols=len(k_gen_list),
    subplot_titles = k_gen_list,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    shared_xaxes=True,
)

for i, k_gen in enumerate(k_gen_list):
    key = jax.random.PRNGKey(SEED)
    z = BATCHED_GENERATORS[k_gen](
        p=p_input,
        key=key,
    )
    state = forward_scan(key, z)
    
    # plot raster of input activity
    fig.add_trace(
        go.Heatmap(
            z=z[0],
            colorscale="gray",
            showscale=False,
            hovertemplate=None,
        ),
        row=1, col=i+1,
    )

    # plot raster of hidden activity
    fig.add_trace(
        go.Heatmap(
            z=state["h"][0],
            colorscale="gray",
            showscale=False,
            hovertemplate=None,
        ),
        row=2, col=i+1,
    )

fig.update_layout(
    height=500,
    width=1000,
    margin=dict(l=0, r=0, t=0, b=0),
    showlegend=False,
)

fig.show()